# Pothole Detection — YOLO11 (Ultralytics) on Kaggle

Fine-tunes **yolo11n.pt** into a single-class `pothole` detector using the
**Potholes Detection YOLOv8** dataset.

**Before running:**
1. *Add Input* (right panel) → search **"Potholes Detection YOLOv8"** → add it (mounts under `/kaggle/input/...`).
2. *Settings* → **Accelerator: GPU** (T4 x2 or P100).
3. *(Optional)* *Settings* → **Internet: On** so the base weights auto-download, **or** add `yolo11n.pt` as an input dataset.
4. **Run All**.

In [ ]:
# Ultralytics ships on Kaggle images; ensure a recent version (YOLO11 needs >=8.3).
import importlib, subprocess, sys
try:
    import ultralytics
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics>=8.3.0'])
    import ultralytics
import torch
ultralytics.checks()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Build a data.yaml that points at the mounted dataset (auto-discovers the folder).
import os, glob, yaml

base = '/kaggle/input/potholes-detection-yolov8'
if not os.path.isdir(base):
    for h in glob.glob('/kaggle/input/*'):
        if os.path.isdir(os.path.join(h, 'train', 'images')):
            base = h; break
print('/kaggle/input =', glob.glob('/kaggle/input/*'))
assert os.path.isdir(base), 'Dataset not found — use Add Input to attach it.'

train_dir = os.path.join(base, 'train', 'images')
val_dir = os.path.join(base, 'valid', 'images')
if not os.path.isdir(val_dir):
    val_dir = os.path.join(base, 'val', 'images')

data = {'train': train_dir, 'val': val_dir, 'nc': 1, 'names': ['pothole']}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)
print(data)
print('train imgs:', len(glob.glob(os.path.join(train_dir, '*'))))
print('val imgs  :', len(glob.glob(os.path.join(val_dir, '*'))))

In [ ]:
# Resolve base weights: use a local/uploaded yolo11n.pt if present, else download.
import glob, os, shutil

def resolve_weights(name='yolo11n.pt'):
    if os.path.exists(name):
        return name
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits:
        shutil.copy(hits[0], name)
        print('using uploaded weights:', hits[0])
        return name
    print(f'[note] {name} not found locally; ultralytics will download it (needs Internet On).')
    return name

WEIGHTS = resolve_weights('yolo11n.pt')
print('base weights:', WEIGHTS)

In [ ]:
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO(WEIGHTS)
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=device,
    project='/kaggle/working/runs/pothole',
    name='yolo11n_potholes',
    seed=0,
    plots=True,
)

In [ ]:
metrics = model.val()
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'precision: {metrics.box.mp:.4f}  recall: {metrics.box.mr:.4f}')

In [ ]:
from IPython.display import Image
Image(filename='/kaggle/working/runs/pothole/yolo11n_potholes/results.png')

In [ ]:
# Visual sanity check on a few validation images.
import glob, os
from IPython.display import Image, display

best = '/kaggle/working/runs/pothole/yolo11n_potholes/weights/best.pt'
pred = YOLO(best)
samples = sorted(glob.glob(os.path.join(val_dir, '*')))[:6]
pred.predict(samples, imgsz=640, conf=0.25, save=True,
             project='/kaggle/working/runs/pothole', name='predict', exist_ok=True)
for p in sorted(glob.glob('/kaggle/working/runs/pothole/predict/*'))[:6]:
    display(Image(filename=p, width=460))

In [ ]:
best = '/kaggle/working/runs/pothole/yolo11n_potholes/weights/best.pt'
onnx_path = YOLO(best).export(format='onnx', opset=12)
print('best.pt :', best)
print('onnx    :', onnx_path)
print('Download these from the Output tab (right panel) after the run finishes.')

## Outputs
- `best.pt` — deploy this (best val mAP).
- `best.onnx` — portable inference (onnxruntime / onnxruntime-web).
- `results.png`, `confusion_matrix.png`, PR/F1 curves — under the run folder.

## Using it in Vouch
The Vouch demo detects road hazards behind a `CameraProvider` interface (`src/camera/`).
To make this model the real perception path, add a provider that runs `best.onnx`
and emits the same `CameraDetection` objects the mock does. See `ml/README.md`.